# Preparation carbon density data cSpatial and cDGVM (figure m01 and m03) 

In [ ]:
# Libraries
import datetime, os
import xarray as xr

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir04 = '../paper_ecosystem_carbon/output/04_dgvm_prep/'
dir05 = '../paper_ecosystem_carbon/output/05_rf_constrain/'
dir07 = '../paper_ecosystem_carbon/output/07_eval_prep/'
dir_out = '../paper_ecosystem_carbon/output/zenodo/'

---

In [ ]:
# Get prepared dgvm data
ds_dgvm = xr.open_zarr(f'{dir04}ds_dgvm_rffill_r025.zarr')
# Get constrained random forest data
ds_rf = xr.open_zarr(f'{dir05}ds_rfconstr_r025.zarr')
# Get prepared occurance data
ds_occur = xr.open_zarr(f'{dir07}ds_prep_occur_r025.zarr')
# Get land mask
ds_land = xr.open_dataset(f'{dir01}ds_luh2_land_mask_r025.nc')

In [ ]:
# Ecosystem categories
dict_etype = dict(trop_ef = 'Tropical/subtropical evergreen forest', 
                  trop_df = 'Tropical/subtropical deciduous forest',
                  temp_ebf = 'Temperate/boreal evergreen broadleaf forest',
                  temp_dbf = 'Temperate/boreal deciduous broadleaf forest',
                  temp_enf = 'Temperate/boreal evergreen needleleaf forest',
                  temp_dnf = 'Temperate/boreal deciduous needleleaf forest',
                  glob_shrub = 'Shrublands',
                  glob_grass = 'Grasslands',
                  glob_tundr = 'Tundra',
                  glob_mount = 'Montane')

In [ ]:
# Prepare and export ecosystem carbon density datasets for zenodo upload

def prep_ds_cspatial_cdgvm(approach):

    """Prepares and exports ecosystem carbon density datasets for zenodo 
    upload"""

    # Select data
    if approach == 'cspatial':
        ds_in = ds_rf
        str_approach = 'cSpatial'
    elif approach == 'cdgvm':
        ds_in = ds_dgvm.mean('model')
        str_approach = 'cDGVM'
    else:
        raise ValueError("approach must be 'cspatial' or 'cdgvm'")

    # Check data grids/coordinates align
    xr.align(ds_in, ds_occur, ds_land, join='exact')
    
    # Create empty dataset
    ds_out = xr.Dataset()

    # Add carbon density variables
    for i in dict_etype.keys():

        # cveg
        ds_out[f'cveg_{i}'] = ds_in[f'cveg_{i}'].where(ds_occur[i]).round(0)
        ds_out[f'cveg_{i}'].attrs = {
            'long_name': (
                'Potential vegetation carbon densities of '
                f'{dict_etype[i].lower()} – {str_approach}'),
            'carbon_pool': 'Vegetation carbon',
            'ecosystem_type': dict_etype[i],
            'approach': str_approach,
            'units': 'tC ha-1'}

        #csoil
        ds_out[f'csoil_{i}'] = ds_in[f'csoil_{i}'].where(ds_occur[i]).round(0)
        ds_out[f'csoil_{i}'].attrs = {
            'long_name': (
                'Potential soil organic carbon densities (0–1 m) of '
                f'{dict_etype[i].lower()} – {str_approach}'),
            'carbon_pool': 'Soil organic carbon (0–1 m)',
            'ecosystem_type': dict_etype[i],
            'approach': str_approach,
            'units': 'tC ha-1',
           }
    
    # Add land mask
    ds_out['land_sea_mask'] = ds_land.luh2_land_mask.astype('int8')
    ds_out['land_sea_mask'].attrs = {
        'long_name': 'land-sea mask',
        'flag_values': [0, 1],
        'flag_meanings': 'sea land'}
    
    # Update latitude and longitude attributes
    ds_out['lat'].attrs = {'long_name': 'latitude',
                           'units': 'degrees_north',
                           'standard_name': 'latitude',
                           'axis': 'Y'}
    
    ds_out['lon'].attrs = {'long_name': 'longitude',
                            'units': 'degrees_east',
                            'standard_name': 'longitude',
                            'axis': 'X'}

    # Update dataset attributes
    date = datetime.datetime.now(datetime.timezone.utc)
    ds_out.attrs.update({
        'title': f'Potential ecosystem carbon densities – {str_approach}',
        'institution': (
            'Department of Geography, '
            'Ludwig-Maximilians-Universität München, Munich, Germany'
        ),
        'creator_name': 'Raphael Ganzenmüller',
        'version': '1.0',
        'date_created': date.strftime('%Y-%m-%d %H:%M UTC')
    })

    # Value range check for int16
    for var in ds_out.data_vars:
        if var != 'land_sea_mask':
            valid = ds_out[var].where(ds_out[var].notnull())
            vmin = valid.min().compute().item()
            vmax = valid.max().compute().item()
            
            assert vmin >= 0, f'{var}: negative value {vmin}'
            assert vmax <= 32767, f'{var}: exceeds int16 range ({vmax})'
    
    # Create directory if missing
    os.makedirs(dir_out, exist_ok=True)
    
    # Get encoding
    encoding = {
        var: {
            'dtype': 'int16',
            '_FillValue': -32768,
            'zlib': True,
            'complevel': 6} 
    for var in ds_out.data_vars
    if var != 'land_sea_mask'
    }

    encoding['land_sea_mask'] = {
        'dtype': 'int8', 
        'zlib': True, 
        'complevel': 6
    }
    
    # Export
    file_out = f'ecosystem_carbon_{approach}_v1-0.nc'
    ds_out.to_netcdf(os.path.join(dir_out, file_out),
                     engine='netcdf4', encoding=encoding, mode='w')


prep_ds_cspatial_cdgvm('cspatial')
prep_ds_cspatial_cdgvm('cdgvm')

---

### Check

In [ ]:
xr.open_dataset(f'{dir_out}ecosystem_carbon_cspatial_v1-0.nc', decode_cf=False)

In [ ]:
xr.open_dataset(f'{dir_out}ecosystem_carbon_cspatial_v1-0.nc', decode_cf=True)

In [ ]:
xr.open_dataset(f'{dir_out}ecosystem_carbon_cdgvm_v1-0.nc', decode_cf=False)

In [ ]:
xr.open_dataset(f'{dir_out}ecosystem_carbon_cdgvm_v1-0.nc', decode_cf=True)

In [ ]:
# Final checks
for i in ['cspatial', 'cdgvm']:
    with xr.open_dataset(
        f'{dir_out}ecosystem_carbon_{i}_v1-0.nc'
    ) as ds:

        assert ds.lat.attrs['axis'] == 'Y'
        assert ds.lon.attrs['axis'] == 'X'
        assert ds.cveg_trop_ef.attrs['units'] == 'tC ha-1'
    
        print(i)
        for var in ds.data_vars:
            print(
                var,
                ds[var].dtype,
                ds[var].min().item(),
                ds[var].max().item()
            )
    
        carbon_vars = [
            v for v in ds.data_vars
            if v.startswith(('cveg_', 'csoil_'))
        ]
        
        assert len(carbon_vars) == 20
        assert ds.sizes['lat'] == 720
        assert ds.sizes['lon'] == 1440
        
        assert ds.land_sea_mask.min().item() == 0
        assert ds.land_sea_mask.max().item() == 1